## GPT annotation results

This notebook calculates recall, precision and accuracy for GPT-4o's annotation on physical locations. It does this by comparing two annotation files:
1. *kohad_1000_gpt_valjund_pipe.csv*: 1000 words automatically tagged as physical locations (LOC) or not (NONE) by GPT-4o. For tagging see file *v03_gpt_annotation.ipynb*
2. *kohasonad_margendus_pipe.csv*: 1000 word manually tagged as physical locations (loc_c), abstract locations (loc_a), 'owner', 'time', 'reason', 'state', 'event', 'dependent','error', 'manner' or 'other'. For detailed description of each class see file *annotation_guidelines_location.md* in folder *documentation*.

Automatic annotation results are cleared of irregularities and unified (ie LOC" and LOC” merged under tag LOC). False positives are counted by tag.

In [1]:
import pandas as pd
from collections import Counter

In [2]:
#words annotated by gpt-4o
gpt_output_path = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\estnltk_syntax_repo_kloon\\physical_location_labelling\\physical_location_by_context\\results\\v03_gpt_annotation\\kohad_1000_gpt_valjund_pipe.csv"
gpt_output = pd.read_csv(gpt_output_path, delimiter = "|")
gpt_output

,lause,form,classification
0,Seevastu tüdrukud on isa Toivo sõnul Markole h...,Markole,NONE
1,“ Lahkusin poliitikast 1999. aastal .,poliitikast,NONE
2,Nüüd käib koheva karvaga kass tähtsal ilmel ai...,ilmel,NONE
3,Kuna nii suurte ruumide koristamine on tööl kä...,Kelamitel,NONE
4,Olen paaril aastavahetusel koolides tasuta jõu...,aastavahetusel,NONE
...,...,...,...
995,mailis: mingit raffast on kes käisid see nädal...,kalevispordihallis,LOC
996,annika: slow sa käid päinakas we,päinakas,NONE
997,WickedSick: ma käis täna auras,auras,LOC
998,BUSTA: keegi RAHZEL il ka käis ?,il,NONE


In [3]:
#manually annotated data
goldstandard_path = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\estnltk_syntax_repo_kloon\\physical_location_labelling\\physical_location_by_context\\results\\v02_manual_annotation\\kohasonad_margendus_pipe.csv"
goldstandard = pd.read_csv(goldstandard_path, delimiter = "|")
goldstandard

,lemma,form,verbifraas,margend,lause,lause_id
0,Marko,Markole,tüdrukud on isa sõnul Markole hoogsalt amokki ...,owner,Seevastu tüdrukud on isa Toivo sõnul Markole h...,775
1,poliitika,poliitikast,Lahkusin poliitikast aastal,loc_a,“ Lahkusin poliitikast 1999. aastal .,880
2,ilme,ilmel,Nüüd käib kass ilmel aias ringi,manner,Nüüd käib koheva karvaga kass tähtsal ilmel ai...,2886
3,Kelam,Kelamitel,raske käib Kelamitel abiks vanadaam,owner,Kuna nii suurte ruumide koristamine on tööl kä...,2898
4,aastavahetus,aastavahetusel,Olen aastavahetusel koolides mängimas käinud,time,Olen paaril aastavahetusel koolides tasuta jõu...,3305
...,...,...,...,...,...,...
995,kalevispordihall,kalevispordihallis,raffast on kes käisid see kalevispordihallis,loc_c,mailis: mingit raffast on kes käisid see nädal...,21138145
996,päinaka,päinakas,slow sa käid päinakas,error,annika: slow sa käid päinakas we,21178354
997,aurama,auras,ma käis täna auras,loc_c,WickedSick: ma käis täna auras,21253362
998,il,il,keegi RAHZEL il ka käis,error,BUSTA: keegi RAHZEL il ka käis ?,21321966


In [4]:
#export annotations to lists for comparison
gold_tag = goldstandard['margend'].tolist()
gpt_tag = gpt_output['classification'].tolist()

In [5]:
#check all unique tags for possible annotation mistakes
print(set(gold_tag))
print(set(gpt_tag))

{'time', 'reason', 'other', 'manner', 'event', 'loc_a', 'error', 'state', 'loc_c', 'owner', 'dependent'}
{'LOC"', 'NONE"', 'NONE', 'LOC”', 'NONE»', 'NONE”', 'LOC'}


In [6]:
#make gpt tags uniform for easier checking
gpt_tag_fixed = []
for tag in gpt_tag:
    if 'LOC' in tag:
        gpt_tag_fixed.append('LOC')
    elif 'NONE' in tag:
        gpt_tag_fixed.append('NONE')
set(gpt_tag_fixed) #check result

{'LOC', 'NONE'}

In [7]:
#count results
TP = 0
FP = 0
TN = 0
FN = 0
FP_tags = [] #what tags were falsely annotated as physical locations

for i, tag in enumerate(gold_tag):
    if tag == 'loc_c': #if gold standard is physical location
        if gpt_tag_fixed[i] == 'LOC': #if gpt tagged it as physical location
            TP += 1
        if gpt_tag_fixed[i] == 'NONE': #if gpt tagged it as something else
            FN += 1
    else: #if gold standard isn't physical location
        if gpt_tag_fixed[i] == 'LOC': #if gpt tagged it as physical location
            FP += 1
            FP_tags.append(tag)
        if gpt_tag_fixed[i] == 'NONE': #if gpt tagged it as not a physical location
            TN += 1

In [8]:
#calculate results
recall = TP/(TP+FN) #how many physical locations got tagged as physical locations
precision = TP/(TP+FP) #how many of the physical location tags were correct
f1 = 2*((precision*recall)/(precision+recall))
accuracy = (TP+TN)/(TP+TN+FP+FN) #how many tags were correct

print("Recall: ", recall)
print("Precision: ", precision)
print("F-score: ", f1)
print("Accuracy: ", accuracy)

Recall:  0.9331306990881459
Precision:  0.7791878172588832
F-score:  0.8492392807745505
Accuracy:  0.891


In [9]:
#how much a tag was wrongly annotated as a physical location 
Counter(FP_tags)

Counter({'loc_a': 52,
         'error': 11,
         'event': 10,
         'other': 10,
         'state': 2,
         'dependent': 1,
         'owner': 1})

In [10]:
print('TP: ', TP)
print('TN: ', TN)
print('FP: ', FP)
print('FN: ', FN)

TP:  307
TN:  584
FP:  87
FN:  22


### Don't count loc_a as error

In [10]:
#count results
TP = 0
FP = 0
TN = 0
FN = 0
FP_tags = [] #what tags were falsely annotated as physical locations

for i, tag in enumerate(gold_tag):
    if tag == 'loc_c': #if gold standard is physical location
        if gpt_tag_fixed[i] == 'LOC': #if gpt tagged it as physical location
            TP += 1
        if gpt_tag_fixed[i] == 'NONE': #if gpt tagged it as something else
            FN += 1
    else: #if gold standard isn't physical location
        if gpt_tag_fixed[i] == 'LOC' and tag != 'loc_a': #if gpt tagged it as physical location
            FP += 1
            FP_tags.append(tag)
        if gpt_tag_fixed[i] == 'NONE': #if gpt tagged it as not a physical location
            TN += 1

In [11]:
#calculate results
recall = TP/(TP+FN) #how many physical locations got tagged as physical locations
precision = TP/(TP+FP) #how many of the physical location tags were correct
f1 = 2*((precision*recall)/(precision+recall))
accuracy = (TP+TN)/(TP+TN+FP+FN) #how many tags were correct

print("Recall: ", recall)
print("Precision: ", precision)
print("F-score: ", f1)
print("Accuracy: ", accuracy)

Recall:  0.9331306990881459
Precision:  0.8976608187134503
F-score:  0.9150521609538003
Accuracy:  0.939873417721519
